# ✂️ StockGen AI - Rembg (Background Removal) Test
สมุดงานนี้ใช้สำหรับทดสอบไลบรารี `rembg` เพื่อลบพื้นหลังออกจากรูปภาพ ก่อนนำไปปรับใช้จริงในระบบ RunPod ของ StockGen AI

**ข้อดีของการรันบน Colab:**
1. ไม่เปลืองพื้นที่เครื่องหลัก (โหลดโมเดลใหญ่ๆ ลง Cloud)
2. ใช้ GPU ฟรี (เช็คว่ารันบน GPU แล้วเร็วแค่ไหน)
3. เปลี่ยนสลับโมเดลทดสอบได้ง่าย (เช่น U2Net, ISNet)

---

### 1. เช็คสเปก GPU (ควรตั้งค่า Runtime เป็น T4 GPU)

In [ ]:
!nvidia-smi

### 2. ติดตั้งไลบรารี `rembg` (รุ่นรองรับ GPU)

In [ ]:
!pip install rembg[gpu]
!pip install pillow requests

### 3. ดาวน์โหลดรูปภาพทดสอบ
เราจะดึงรูปตัวอย่างมาทดสอบลบพื้นหลัง

In [ ]:
import requests
from PIL import Image
from io import BytesIO

# โหลดรูปจากเน็ต (เปลี่ยน URL ได้ตามต้องการ)
image_url = 'https://raw.githubusercontent.com/danielgatis/rembg/master/examples/animal-1.jpg'
response = requests.get(image_url)
input_image = Image.open(BytesIO(response.content))

print("📥 ภาพต้นฉบับโหลดเสร็จแล้ว ขนาด:", input_image.size)
display(input_image.resize((300, int(300 * input_image.height / input_image.width))))

### 4. ทดสอบลบพื้นหลังด้วยโมเดลมาตรฐาน (`u2net`)
โมเดล `u2net` คือตัวเริ่มต้น (Default) ซึ่งทำงานได้ดีครอบคลุมเกือบทุกประเภท

In [ ]:
import time
from rembg import remove, new_session

print("🔄 กำลังโหลดโมเดล u2net (โหลดครั้งแรกอาจใช้เวลานิดหน่อย)...")
# สร้าง Session เพื่อให้มันโหลดโมเดลเก็บไว้ใน RAM
session_u2net = new_session('u2net')

print("✂️ กำลังลบพื้นหลัง...")
start_time = time.time()

# สั่งลบพื้นหลัง (ส่ง PIL เข้าไป จะได้ PIL คืนมาเป็น RGBA พื้นใส)
output_image = remove(input_image, session=session_u2net)

end_time = time.time()
print(f"✅ ลบเสร็จเรียบร้อย! ใช้เวลาประมวลผล: {end_time - start_time:.2f} วินาที")

# สร้างพื้นหลังสีหมากรุก (Checkerboard) เพื่อให้เห็นความโปร่งใสชัดๆ
def display_with_checkerboard(img):
    from PIL import ImageDraw
    bg = Image.new('RGB', img.size, (255, 255, 255))
    draw = ImageDraw.Draw(bg)
    sq_size = 20
    for y in range(0, img.height, sq_size):
        for x in range(0, img.width, sq_size):
            if (x // sq_size + y // sq_size) % 2 == 0:
                draw.rectangle([x, y, x+sq_size, y+sq_size], fill=(200, 200, 200))
    bg.paste(img, (0, 0), img)
    display(bg.resize((300, int(300 * bg.height / bg.width))))

display_with_checkerboard(output_image)

### 5. (ทางเลือก) ทดสอบลบด้วย Alpha Matting
การเปิด Alpha Matting จะช่วยให้ระบบคำนวณขอบที่ซับซ้อน (เช่น เส้นขน, ผม, ขอบใบไม้) ให้เนียนขึ้น แต่อาจใช้เวลาประมวลผลนานขึ้นเล็กน้อย

In [ ]:
print("✂️ กำลังลบพื้นหลัง (เปิดโหมด Alpha Matting)...")
start_time = time.time()

output_matting = remove(
    input_image, 
    session=session_u2net,
    alpha_matting=True,
    alpha_matting_foreground_threshold=240,
    alpha_matting_background_threshold=10,
    alpha_matting_erode_size=10
)

end_time = time.time()
print(f"✅ ลบเสร็จเรียบร้อย! ใช้เวลาประมวลผล: {end_time - start_time:.2f} วินาที")

display_with_checkerboard(output_matting)